# Hate speech classification

TF-IDF features and Gaussian Naive Bayes on the public Davidson dataset. The implementation lives in `classifier.py`.

Install `requirements.txt`, select the project `.venv`, and run all cells from the repository directory. The source data contains offensive language.

In [ ]:
import pandas as pd

from classifier import download, load_dataset, train

dataset = download()
data, cleaning = load_dataset(dataset)
pd.Series(cleaning, name="Rows and groups")

## Data

Original hate-speech class `0` maps to `1`; offensive language and neither map to `0`. Non-hate does not mean harmless. Conflicting and repeated texts are removed before splitting; IDs and annotation votes are excluded from model input.

In [ ]:
data["label"].value_counts().rename(index={0: "Non-hate", 1: "Hate"}).to_frame(
    "Posts"
)

## Train and evaluate

Reserve 20% for evaluation, preserving class proportions (seed 42). Five-fold cross-validation selects smoothing by macro F1, weighting both classes equally. TF-IDF is fitted inside each fold, capped at 2,000 features.

This takes a few minutes. Evaluation is saved before refitting on all cleaned rows for prediction.

In [ ]:
report = train(dataset)
pd.DataFrame(
    {
        name: {
            "Accuracy": report[key]["accuracy"],
            "Macro F1": report[key]["macro_f1"],
            "Hate precision": report[key]["classification_report"]["hate"][
                "precision"
            ],
            "Hate recall": report[key]["classification_report"]["hate"][
                "recall"
            ],
        }
        for name, key in [
            ("Gaussian NB", "holdout"),
            ("Majority baseline", "majority_baseline"),
        ]
    }
).T

## Errors

Rows are actual labels; columns are predictions. Hate precision measures how many flags are correct; recall measures how much labeled hate the model catches.

In [ ]:
pd.DataFrame(
    report["holdout"]["confusion_matrix"],
    index=["Actual non-hate", "Actual hate"],
    columns=["Predicted non-hate", "Predicted hate"],
)

## Use and limitations

```text
python classifier.py predict --input posts.csv --output artifacts/predictions.csv
```

Input: `id,tweet`. Output: `id,label`. Metrics and versions: `artifacts/metrics.json`.

Gaussian assumptions are a poor fit for sparse text features. This historical dataset has subjective labels and documented bias; results do not establish suitability for moderation.

[Dataset](https://github.com/t-davidson/hate-speech-and-offensive-language) | [Paper](https://arxiv.org/abs/1703.04009) | [Bias study](https://arxiv.org/abs/1905.12516). See the README for setup and attribution.